# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying loss

## load data

In [132]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_loss'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'auxi_type', 'auxi_mode', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        # shutil.rmtree(exp_dir)
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    # if np.isnan(metric).any():
    #     shutil.rmtree(exp_dir)
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

## preprocess

In [133]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes = pd.read_csv(f'{save_root}/best_finetune_full_each.csv')

best = finetunes.copy()
best = best[best.data_id.isin(['ETTm1_PCA', 'ETTh1_PCA', 'ECL_PCA', 'Weather_PCA'])]
best = best[best.model.isin(['Fredformer', 'iTransformer'])]
best['data_id'] = best['data_id'].replace({'ECL_PCA': 'ECL', 'Weather_PCA': 'Weather', 'ETTm1_PCA': 'ETTm1', 'ETTh1_PCA': 'ETTh1'})
best = best[best.pred_len != 'Avg']

base = baselines.copy()
base = base[base.data_id.isin(['ETTm1', 'ETTh1', 'ECL', 'Weather'])]
base = base[base.model.isin(['Fredformer', 'iTransformer'])]
base['auxi_mode'] = 'DF'


df2 = df.copy()
min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len', 'auxi_mode'])['mse'].idxmin()
df2 = df2.loc[min_mse_idx]
df2 = df2[df2.model.isin(['Fredformer', 'iTransformer'])]


columns = ['model', 'pred_len', 'data_id', 'auxi_mode', 'mse', 'mae']
df_loss = pd.concat([df2[columns], best[columns], base[columns]], ignore_index=True)
df_loss['pred_len'] = df_loss['pred_len'].astype(int)
df_loss.replace({'auxi_mode': {'basis': 'PDF', 'rfft': 'FreDF', 'fourier_koopman': 'Fourier Koopman', 'dpp': 'DPP', 'dilate_cuda': 'Dilate', 'soft_dtw': 'Soft-DTW', 'dtw': 'DTW'}}, inplace=True)

df_loss = df_loss[df_loss['auxi_mode'].isin(['DF', 'PDF', 'FreDF', 'Fourier Koopman', 'Dilate', 'Soft-DTW', 'DTW'])]

dst_order = ['ETTm1', 'ETTh1', 'ECL', 'Weather']
df_loss['data_id'] = pd.Categorical(df_loss['data_id'], categories=dst_order, ordered=True)

mode_order = ['DF', 'PDF', 'FreDF', 'Fourier Koopman', 'Dilate', 'Soft-DTW', 'DTW']
df_loss['auxi_mode'] = pd.Categorical(df_loss['auxi_mode'], categories=mode_order, ordered=True)


df_loss_avg = df_loss.groupby(['model', 'data_id', 'auxi_mode']).mean(numeric_only=True).reset_index()
df_loss_avg['pred_len'] = 'Avg'
df_loss = pd.concat([df_loss, df_loss_avg]).reset_index(drop=True)
df_loss.sort_values(by=['model', 'data_id', 'auxi_mode', 'pred_len'], inplace=True)
df_loss.dropna(inplace=True, thresh=6)


# save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
# save_cols = ['data_id', 'pred_len', 'mse', 'mae', 'auxi_mode']
# df_loss.round(3)[save_cols].to_csv(f'{save_root}/diff_loss.csv', index=False, float_format='%.3f')
df_loss.head(4)

/tmp/ipykernel_901119/3935926646.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_loss_avg = df_loss.groupby(['model', 'data_id', 'auxi_mode']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,auxi_mode,mse,mae
200,Fredformer,96,ETTm1,DF,0.326369,0.360869
201,Fredformer,192,ETTm1,DF,0.365194,0.382132
202,Fredformer,336,ETTm1,DF,0.395987,0.404369
203,Fredformer,720,ETTm1,DF,0.459217,0.444342


In [135]:
df_loss_it = df_loss[df_loss['model'] == 'iTransformer'].copy()

df_loss_it = df_loss_it.set_index(['data_id', 'pred_len', 'auxi_mode']).unstack('auxi_mode').swaplevel(axis=1)
columns = []
for model in df_loss_it.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df_loss_it = df_loss_it[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df_loss_it.to_csv(f'{save_root}/diff_loss_itransformer.csv', index=True, float_format='%.3f')

df_loss_it

auxi_mode               DF                 PDF               FreDF            \
                       mse       mae       mse       mae       mse       mae   
data_id pred_len                                                               
ETTm1   96        0.337877  0.372283  0.323290  0.358108  0.334168  0.364617   
        192       0.381666  0.396060  0.371106  0.388443  0.381001  0.389560   
        336       0.426993  0.423962  0.407612  0.406701  0.417425  0.411991   
        720       0.495750  0.462558  0.477105  0.449605  0.489288  0.452975   
        Avg       0.410572  0.413716  0.394778  0.400714  0.405470  0.404786   
ETTh1   96        0.385231  0.405311  0.377988  0.393197  0.378290  0.394857   
        192       0.440491  0.436761  0.428025  0.422552  0.428218  0.422927   
        336       0.479858  0.457185  0.472883  0.450154  0.470420  0.446535   
        720       0.503538  0.491569  0.472567  0.468893  0.490115  0.484076   
        Avg       0.452279  0.447707  0.437866  0.433699  0.441761  0.437099   
ECL     96        0.150035  0.241510  0.144906  0.234782  0.149318  0.238385   
        192       0.168114  0.259067  0.158948  0.248666  0.163185  0.251200   
        336       0.182346  0.274355  0.173076  0.264487  0.179219  0.268179   
        720       0.214451  0.303504  0.203276  0.292044  0.211727  0.297087   
        Avg       0.178736  0.269609  0.170051  0.259995  0.175862  0.263713   
Weather 96        0.171432  0.210468  0.162750  0.201556  0.170192  0.208080   
        192       0.246395  0.278306  0.213842  0.247811  0.219245  0.252216   
        336       0.296230  0.312968  0.273705  0.293638  0.279228  0.295746   
        720       0.362297  0.352779  0.351255  0.344143  0.357757  0.347267   
        Avg       0.269089  0.288630  0.250388  0.271787  0.256605  0.275827   

auxi_mode        Fourier Koopman              Dilate            Soft-DTW  \
                             mse       mae       mse       mae       mse   
data_id pred_len                                                           
ETTm1   96              0.350220  0.382423  0.341916  0.376365  0.339033   
        192             0.388839  0.399998  0.380547  0.395505  0.382808   
        336             0.424593  0.422521  0.417720  0.418465  0.429142   
        720             0.488876  0.458327  0.487312  0.457452  0.515738   
        Avg             0.413132  0.415817  0.406874  0.411946  0.416680   
ETTh1   96              0.391786  0.411401  0.385230  0.405311  0.387440   
        192             0.445762  0.442348  0.440491  0.436761  0.442768   
        336             0.482555  0.460535  0.479858  0.457185  0.493801   
        720             0.501223  0.490956  0.503536  0.491568  0.557050   
        Avg             0.455332  0.451310  0.452279  0.447706  0.470265   
ECL     96              0.150679  0.242527  0.149644  0.241297  0.149093   
        192             0.167157  0.257469  0.168034  0.259151  0.164025   
        336             0.182492  0.275084  0.181248  0.274189  0.180230   
        720             0.211651  0.299635  0.211688  0.299738  0.206831   
        Avg             0.177995  0.268679  0.177654  0.268594  0.175045   
Weather 96              0.206135  0.256812  0.208428  0.258807  0.206857   
        192             0.263646  0.299696  0.251624  0.284944  0.264086   
        336             0.308673  0.326256  0.310589  0.327818  0.313679   
        720             0.377120  0.368881  0.373645  0.363635  0.384204   
        Avg             0.288894  0.312911  0.286071  0.308801  0.292206   

auxi_mode                        DTW            
                       mae       mse       mae  
data_id pred_len                                
ETTm1   96        0.373351  0.341274  0.375217  
        192       0.394607  0.383154  0.395302  
        336       0.422968  0.429267  0.422762  
        720       0.468649  0.512092  0.467383  
        Avg       0.414894  0.416447  0.415166  
ETTh1   96        

In [126]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'auxi_mode', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'batch_size', 'lradj', 'patience', 'train_epochs']
data_id = 'ETTm1'
pl = 720
df[(df['model'] == 'iTransformer') & (df['data_id'] == 'ECL') & (df['auxi_mode'].isin(['soft_dtw', 'dtw']) & (df.pred_len == pl))].sort_values(by=['auxi_mode', 'pred_len', 'mse'])[columns]

,model,pred_len,data_id,mse,mae,auxi_mode,learning_rate,rec_lambda,auxi_lambda,batch_size,lradj,patience,train_epochs
204,iTransformer,720,ECL,0.211628,0.299558,dtw,0.0010,0.999995,0.000005,32,type1,3,10
22,iTransformer,720,ECL,0.373097,0.398250,dtw,0.0010,0.999950,0.000050,32,type1,3,10
198,iTransformer,720,ECL,1.316669,0.850759,dtw,0.0005,0.999500,0.000500,32,type1,10,10
71,iTransformer,720,ECL,1.321301,0.856929,dtw,0.0005,0.999500,0.000500,32,type1,3,10
14,iTransformer,720,ECL,1.348461,0.867783,dtw,0.0010,0.999500,0.000500,32,type1,3,10
114,iTransformer,720,ECL,1.348461,0.867783,dtw,0.0010,0.999500,0.000500,32,type1,10,10
250,iTransformer,720,ECL,1.503271,0.939518,dtw,0.0010,0.995000,0.005000,32,type1,3,10
188,iTransformer,720,ECL,1.527094,0.947609,dtw,0.0010,0.000000,1.000000,32,type1,3,10
268,iTransformer,720,ECL,0.206831,0.295923,soft_dtw,0.0020,0.999995,0.000005,32,type1,3,10
18,iTransformer,720,ECL,0.211091,0.299552,soft_dtw,0.0010,0.999995,0.000005,32,type1,3,10


In [131]:
df_loss_fr = df_loss[df_loss['model'] == 'Fredformer'].copy()

df_loss_fr = df_loss_fr.set_index(['data_id', 'pred_len', 'auxi_mode']).unstack('auxi_mode').swaplevel(axis=1)
columns = []
for model in df_loss_fr.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df_loss_fr = df_loss_fr[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df_loss_fr.to_csv(f'{save_root}/diff_loss_fredformer.csv', index=True, float_format='%.3f')

df_loss_fr

auxi_mode               DF                 PDF               FreDF            \
                       mse       mae       mse       mae       mse       mae   
data_id pred_len                                                               
ETTm1   96        0.326369  0.360869  0.321138  0.357436  0.326345  0.354604   
        192       0.365194  0.382132  0.359581  0.378119  0.363325  0.379993   
        336       0.395987  0.404369  0.389238  0.399987  0.392050  0.399722   
        720       0.459217  0.444342  0.447020  0.434891  0.454542  0.439892   
        Avg       0.386692  0.397928  0.379244  0.392608  0.384066  0.393553   
ETTh1   96        0.377193  0.395888  0.368003  0.390765  0.369706  0.391559   
        192       0.437019  0.425390  0.424089  0.421955  0.435593  0.436542   
        336       0.485769  0.448580  0.466960  0.441356  0.472972  0.442835   
        720       0.487772  0.467352  0.464932  0.463144  0.474102  0.465845   
        Avg       0.446938  0.434302  0.430996  0.429305  0.438093  0.434195   
ECL     96        0.161018  0.257591  0.151151  0.244797  0.152477  0.247149   
        192       0.173585  0.269470  0.165649  0.256300  0.166204  0.257184   
        336       0.194108  0.290397  0.181096  0.274290  0.182871  0.277835   
        720       0.234672  0.319384  0.213155  0.304079  0.216380  0.304361   
        Avg       0.190846  0.284211  0.177763  0.269866  0.179483  0.271632   
Weather 96        0.180192  0.219763  0.171210  0.208291  0.173795  0.212956   
        192       0.222379  0.257628  0.219460  0.252813  0.219494  0.253721   
        336       0.283492  0.300976  0.276752  0.295328  0.278017  0.296093   
        720       0.357656  0.348500  0.353255  0.346356  0.353531  0.346543   
        Avg       0.260930  0.281717  0.255169  0.275697  0.256209  0.277328   

auxi_mode        Fourier Koopman              Dilate            Soft-DTW  \
                             mse       mae       mse       mae       mse   
data_id pred_len                                                           
ETTm1   96              0.335296  0.367959  0.336542  0.366889  0.332027   
        192             0.365785  0.384240  0.363552  0.383585  0.370162   
        336             0.398761  0.408379  0.397105  0.405789  0.406413   
        720             0.456066  0.441159  0.456893  0.442939  0.478230   
        Avg             0.388977  0.400434  0.388523  0.399801  0.396708   
ETTh1   96              0.374758  0.397324  0.377725  0.398754  0.375807   
        192             0.437866  0.433680  0.439037  0.434780  0.439461   
        336             0.473031  0.455418  0.480581  0.452517  0.484394   
        720             0.522900  0.487064  0.515930  0.482110  0.541657   
        Avg             0.452139  0.443371  0.453318  0.442040  0.460330   
ECL     96              0.165501  0.263157  0.158221  0.253437  0.167922   
        192             0.173701  0.266879  0.170281  0.263345  0.218242   
        336             0.187510  0.280454  0.190351  0.285521  0.197202   
        720             0.231759  0.317802  0.229186  0.316376  0.239800   
        Avg             0.189618  0.282073  0.187010  0.279670  0.205792   
Weather 96              0.173798  0.214170  0.172911  0.213653  0.173437   
        192             0.220210  0.255959  0.225348  0.259521  0.220364   
        336             0.279564  0.298372  0.279500  0.298914  0.281006   
        720             0.354267  0.347410  0.355271  0.347889  0.369443   
        Avg             0.256960  0.278978  0.258258  0.279994  0.261063   

auxi_mode                        DTW            
                       mae       mse       mae  
data_id pred_len                                
ETTm1   96        0.363381  0.331829  0.364233  
        192       0.385921  0.369529  0.386404  
        336       0.408827  0.408648  0.410411  
        720       0.449607  0.475852  0.448433  
        Avg       0.401934  0.396464  0.402370  
ETTh1   96        